# Dishmook — مقایسهٔ Campaignها پس از انتخاب مدل

این Notebook هنوز روی Kaggle اجرا نشده است. فقط یک نشست GPU رایگان و دارای سهمیه انتخاب کن؛ هیچ ارتقای پولی لازم نیست. اینترنت فقط برای نصب و دریافت وزن‌های عمومی در مرحلهٔ آماده‌سازی لازم است. اجرای مدل محلی است. پیش از پایان نشست، آرشیو شواهد را دانلود کن.

Use a reviewed source ZIP or pinned Git commit. This notebook does not bypass quotas, create accounts or provision paid services. Python/runtime, disk and GPU availability are checked explicitly. Model quality is not assumed.


In [ ]:
from pathlib import Path
import json, re, subprocess, sys, zipfile
PROJECT = Path('/kaggle/working/Dishmook')
SOURCE_ARCHIVE = Path('/kaggle/input/dishmook-source/Dishmook-source.zip')
SOURCE_REVISION = None  # Set to a reviewed Git commit if using GitHub instead of the attached source ZIP.
if not PROJECT.exists():
    if SOURCE_ARCHIVE.is_file():
        with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
            for item in archive.namelist():
                if item.startswith('/') or '..' in Path(item).parts or '\\' in item:
                    raise ValueError('Unsafe source archive')
            archive.extractall('/kaggle/working')
    else:
        if not SOURCE_REVISION or not re.fullmatch('[a-f0-9]{40}', SOURCE_REVISION):
            raise RuntimeError('Attach Dishmook-source.zip as a Kaggle input, or set SOURCE_REVISION.')
        subprocess.run(['git','clone','https://github.com/amin076/Dishmook.git',str(PROJECT)],check=True)
        subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_REVISION],check=True)
if sys.version_info[:2] != (3,12):
    raise RuntimeError('This release is tested on Python 3.12. Select a compatible runtime before installing.')
subprocess.run([sys.executable,'-m','pip','install','-e',str(PROJECT)+'[dev,gpu]'],check=True)
def cli(*args):
    subprocess.run([sys.executable,'-m','dishmook',*map(str,args)],cwd=PROJECT,check=True)
cli('gpu-preflight')  # Stop if no free GPU session is available; there is no paid fallback.


In [ ]:
WORK = Path('/kaggle/working/dishmook-campaigns')
WORK.mkdir(exist_ok=True)
SELECTION = Path('/kaggle/input/dishmook-evidence/model_selection.json')
selection = json.loads(SELECTION.read_text())
if selection.get('status') != 'provisional_selection' or not selection.get('winner'):
    raise RuntimeError('Real model selection evidence is missing or inconclusive. Do not substitute a fake winner.')
selected = selection['winner']['model']
catalog = json.loads((PROJECT/'configs/model_candidates.json').read_text())
name = next(c['name'] for c in catalog['candidates'] if c['model_id']==selected['model_id'] and c['revision']==selected['revision'])
model_config = WORK/'selected_model.json'
cli('prepare-model',name,'--catalog',PROJECT/'configs/model_candidates.json',
    '--models-dir','/kaggle/working/models','--download','--output',model_config)
# Equal total ceilings across 1/self-reflection/5/10/20/50 profiles, recorded actual usage.
# Limit each invocation so a free session can be checkpointed. Re-run to continue.
cli('evaluate','--model',model_config,'--output-dir',WORK/'comparison',
    '--seeds','7','19','41','--max-new-campaigns','6')


In [ ]:
# Run only after the preceding command has stopped. Download the ZIP from Kaggle outputs.
cli('archive-checkpoint',WORK,'/kaggle/working/Dishmook-evidence.zip')


## ادامهٔ اجرا

برای بازیابی، آرشیو شواهد را به نشست بعدی اضافه کن و `dishmook restore-checkpoint ARCHIVE NEW_DIRECTORY` را اجرا کن. مسیر ذخیره‌شدهٔ مدل باید به همان snapshot تغییرناپذیر اشاره کند؛ در این نسخه جابه‌جایی مسیر مدل برای یک اجرای ناتمام پشتیبانی نمی‌شود. همان ساختار مسیر `/kaggle/working/models` را بازسازی کن. وزن‌ها در آرشیو checkpoint ذخیره نمی‌شوند.
